**Data Cleaning and Prearation**

*Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation
before analysis. This phase prepares the dataset for reliable analytics.*

In [28]:
import pandas as pd

In [29]:
folder = r"C:\Users\khush\idx files"
reports_folder = r"C:\Users\khush\Desktop\IDX-Exchange\Reports"

sold = pd.read_csv(reports_folder + r"\sold_with_rates.csv", low_memory=False)
listings = pd.read_csv(reports_folder + r"\listings_with_rates.csv", low_memory=False)

In [30]:
#starting row counts before
print(f"Sold: {sold.shape[0]:,} rows x {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

Sold: 448,253 rows x 84 columns
Listings: 607,724 rows x 86 columns


## Step 1 – Convert Date Columns to Datetime
Date columns are stored as strings by default. Converting them to datetime 
allows us to sort by date, calculate time differences, and flag inconsistencies.

In [31]:
#sold data conversions
date_cols_sold = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
sold[date_cols_sold] = sold[date_cols_sold].apply(pd.to_datetime, errors='coerce')

print("Sold date dtypes:")
print(sold[date_cols_sold].dtypes)

Sold date dtypes:
CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object


In [32]:
#listing data conversions
date_cols_listings = ['ListingContractDate', 'ContractStatusChangeDate']
listings[date_cols_listings] = listings[date_cols_listings].apply(pd.to_datetime, errors='coerce')

print("Listings date dtypes:")
print(listings[date_cols_listings].dtypes)

Listings date dtypes:
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object


In [33]:
#adding month and year columns
sold['Month'] = pd.to_datetime(sold['year_month'], format='%Y-%m', errors='coerce').dt.month
sold['Year'] = pd.to_datetime(sold['year_month'], format='%Y-%m', errors='coerce').dt.year
listings['Month'] = pd.to_datetime(listings['year_month'], format='%Y-%m', errors='coerce').dt.month
listings['Year'] = pd.to_datetime(listings['year_month'], format='%Y-%m', errors='coerce').dt.year

print(sold[['year_month', 'Month', 'Year']].head())

  year_month  Month  Year
0    2024-01      1  2024
1    2024-01      1  2024
2    2024-01      1  2024
3    2024-01      1  2024
4    2024-01      1  2024


*Remove unnecessary columns*

In [34]:
cols_to_drop = [
    # 100% empty columns
    'TaxAnnualAmount', 'AboveGradeFinishedArea', 'TaxYear',
    'ElementarySchoolDistrict', 'CoveredSpaces', 'BusinessType',
    'MiddleOrJuniorSchoolDistrict', 'FireplacesTotal',
    # >90% missing and not useful for analysis
    'WaterfrontYN', 'BelowGradeFinishedArea', 'BuilderName',
    'LotSizeDimensions', 'BuildingAreaTotal', 'CoBuyerAgentFirstName'
] #we kept BasementYN

sold = sold.drop(columns=cols_to_drop, errors='ignore')
listings = listings.drop(columns=cols_to_drop, errors='ignore')

print(f"Sold: {sold.shape[0]:,} rows x {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

Sold: 448,253 rows x 72 columns
Listings: 607,724 rows x 75 columns


## Step 3 – Drop Redundant and Metadata Columns
Removing columns that are duplicates, purely administrative, or not needed for market analysis.

In [35]:
sold.columns

Index(['Flooring', 'ViewYN', 'BasementYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate',
  

In [36]:
listings.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric',
       'MLSAreaMajor', 'CountyOrParish', 'PropertyType.1', 'MlsStatus',
       'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN',
       'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName',
       'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1',
       'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation',
       'BedroomsTotal', 'ContractStatusChangeDate', 'Lo

In [47]:
# Drop .1 duplicate columns from listings (these are exact duplicates created during the combine step)
dot1_cols = [col for col in listings.columns if col.endswith('.1')]
print("Duplicate .1 columns found in listings:", dot1_cols)
listings = listings.drop(columns=dot1_cols, errors='ignore')
print(f"Listings after dropping .1 columns: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

Duplicate .1 columns found in listings: []
Listings after dropping .1 columns: 607,724 rows x 57 columns


In [49]:
redundant_cols = [
    'LotSizeArea', 'LotSizeSquareFeet',          # redundant with LotSizeAcres
    'ListingKeyNumeric', 'ListingId',             # redundant with ListingKey
    'BuyerAgencyCompensationType', 'BuyerAgencyCompensation',  # not needed
    'OriginatingSystemName', 'OriginatingSystemSubName',       # admin metadata
    'StreetNumberNumeric',                        # redundant with UnparsedAddress
    'MlsStatus',                                  # always Closed in sold data
    'PropertyType',                               # always Residential
    'CoListAgentFirstName', 'CoListAgentLastName', # co-agent metadata
    'BuyerAgentMlsId',                            # admin metadata
    'AssociationFeeFrequency',                    # less useful than AssociationFee
    'SubdivisionName',                            # not needed for market analysis
]

sold = sold.drop(columns=redundant_cols, errors='ignore')
listings = listings.drop(columns=redundant_cols, errors='ignore')

print(f"Sold: {sold.shape[0]:,} rows x {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

Sold: 448,253 rows x 67 columns
Listings: 607,724 rows x 57 columns


## Step 4 – Flag Invalid Numeric Values
Rather than deleting these records, we flag them so they can be excluded 
from specific analyses while keeping the raw data intact.

In [51]:
# Sold invalid numeric flags
sold['invalid_close_price'] = sold['ClosePrice'] <= 0
sold['invalid_living_area'] = sold['LivingArea'] <= 0
sold['invalid_days_on_market'] = sold['DaysOnMarket'] < 0
sold['invalid_bed_or_bath'] = (sold['BedroomsTotal'] < 0) | (sold['BathroomsTotalInteger'] < 0)

# Summary
print("=== Invalid Numeric Flags (Sold) ===")
print(f"Invalid ClosePrice: {sold['invalid_close_price'].sum()}")
print(f"Invalid LivingArea: {sold['invalid_living_area'].sum()}")
print(f"Invalid DaysOnMarket: {sold['invalid_days_on_market'].sum()}")
print(f"Invalid Bed/Bath: {sold['invalid_bed_or_bath'].sum()}")



=== Invalid Numeric Flags (Sold) ===
Invalid ClosePrice: 1
Invalid LivingArea: 165
Invalid DaysOnMarket: 51
Invalid Bed/Bath: 0


In [40]:
listings["invalid_close_price"] = listings["ClosePrice"] <= 0
listings["invalid_living_area"] = listings["LivingArea"] <= 0
listings["invalid_days_on_market"] = listings["DaysOnMarket"] < 0
listings["invalid_bed_or_bath"] = (listings["BathroomsTotalInteger"] < 0) | (listings["BedroomsTotal"] < 0)

print("Invalid ClosePrice:", listings["invalid_close_price"].sum())
print("Invalid LivingArea:", listings["invalid_living_area"].sum())
print("Invalid DaysOnMarket:", listings["invalid_days_on_market"].sum())
print("Invalid Bed/Bath:", listings["invalid_bed_or_bath"].sum())

Invalid ClosePrice: 0
Invalid LivingArea: 393
Invalid DaysOnMarket: 32
Invalid Bed/Bath: 0


In [52]:
# Listings invalid numeric flags
listings['invalid_living_area'] = listings['LivingArea'] <= 0
listings['invalid_days_on_market'] = listings['DaysOnMarket'] < 0
listings['invalid_bed_or_bath'] = (listings['BedroomsTotal'] < 0) | (listings['BathroomsTotalInteger'] < 0)

print("\n=== Invalid Numeric Flags (Listings) ===")
print(f"Invalid LivingArea: {listings['invalid_living_area'].sum()}")
print(f"Invalid DaysOnMarket: {listings['invalid_days_on_market'].sum()}")
print(f"Invalid Bed/Bath: {listings['invalid_bed_or_bath'].sum()}")


=== Invalid Numeric Flags (Listings) ===
Invalid LivingArea: 393
Invalid DaysOnMarket: 32
Invalid Bed/Bath: 0


## Step 5 – Date Consistency Flags
Validate the logical order of date fields. ListingContractDate should precede 
PurchaseContractDate, which should precede CloseDate. Flag any violations 
for later review.

In [53]:
# Sold date consistency flags
sold['listing_after_close_flag'] = sold['ListingContractDate'] > sold['CloseDate']
sold['purchase_after_close_flag'] = sold['PurchaseContractDate'] > sold['CloseDate']
sold['negative_timeline_flag'] = sold['ListingContractDate'] > sold['PurchaseContractDate']

print("=== Date Consistency Flags (Sold) ===")
print(f"Listing after close: {sold['listing_after_close_flag'].sum()}")
print(f"Purchase after close: {sold['purchase_after_close_flag'].sum()}")
print(f"Negative timeline: {sold['negative_timeline_flag'].sum()}")

=== Date Consistency Flags (Sold) ===
Listing after close: 70
Purchase after close: 239
Negative timeline: 295


## Step 6 – Geographic Data Quality Flags
Flag records with missing, invalid, or out-of-state coordinates.
California longitudes should always be negative and within the bounding box
of roughly 32°N–42°N latitude and -125°W to -114°W longitude.

In [54]:
# Sold geographic flags
sold['missing_coordinates'] = sold['Latitude'].isnull() | sold['Longitude'].isnull()
sold['zero_coordinates'] = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
sold['invalid_longitude'] = sold['Longitude'] > 0
sold['out_of_state_flag'] = sold['StateOrProvince'] != 'CA'
sold['out_of_bounds_flag'] = (
    ~sold['Latitude'].between(32.529508, 42.009503) | 
    ~sold['Longitude'].between(-124.482003, -114.131211)
).fillna(False)

# Listings geographic flags
listings['missing_coordinates'] = listings['Latitude'].isnull() | listings['Longitude'].isnull()
listings['zero_coordinates'] = (listings['Latitude'] == 0) | (listings['Longitude'] == 0)
listings['invalid_longitude'] = listings['Longitude'] > 0
listings['out_of_state_flag'] = listings['StateOrProvince'] != 'CA'
listings['out_of_bounds_flag'] = (
    ~listings['Latitude'].between(32.529508, 42.009503) | 
    ~listings['Longitude'].between(-124.482003, -114.131211)
).fillna(False)

print("=== Geographic Flags (Sold) ===")
print(f"Missing coordinates: {sold['missing_coordinates'].sum()}")
print(f"Zero coordinates: {sold['zero_coordinates'].sum()}")
print(f"Invalid longitude: {sold['invalid_longitude'].sum()}")
print(f"Out of state: {sold['out_of_state_flag'].sum()}")
print(f"Out of bounds: {sold['out_of_bounds_flag'].sum()}")

print("\n=== Geographic Flags (Listings) ===")
print(f"Missing coordinates: {listings['missing_coordinates'].sum()}")
print(f"Zero coordinates: {listings['zero_coordinates'].sum()}")
print(f"Invalid longitude: {listings['invalid_longitude'].sum()}")
print(f"Out of state: {listings['out_of_state_flag'].sum()}")
print(f"Out of bounds: {listings['out_of_bounds_flag'].sum()}")

=== Geographic Flags (Sold) ===
Missing coordinates: 4388
Zero coordinates: 37
Invalid longitude: 31
Out of state: 30
Out of bounds: 4492

=== Geographic Flags (Listings) ===
Missing coordinates: 80824
Zero coordinates: 75
Invalid longitude: 92
Out of state: 236
Out of bounds: 81184


## Step 7 – Summary of Flags and Row Counts

In [55]:
print("=== Final Shape ===")
print(f"Sold: {sold.shape[0]:,} rows x {sold.shape[1]} columns")
print(f"Listings: {listings.shape[0]:,} rows x {listings.shape[1]} columns")

print("\n=== Flag Summary (Sold) ===")
flag_cols_sold = [col for col in sold.columns if col.endswith('_flag')]
for col in flag_cols_sold:
    print(f"{col}: {sold[col].sum()}")

print("\n=== Flag Summary (Listings) ===")
flag_cols_listings = [col for col in listings.columns if col.endswith('_flag')]
for col in flag_cols_listings:
    print(f"{col}: {listings[col].sum()}")

=== Final Shape ===
Sold: 448,253 rows x 68 columns
Listings: 607,724 rows x 59 columns

=== Flag Summary (Sold) ===
listing_after_close_flag: 70
purchase_after_close_flag: 239
negative_timeline_flag: 295
out_of_state_flag: 30
out_of_bounds_flag: 4492

=== Flag Summary (Listings) ===
out_of_state_flag: 236
out_of_bounds_flag: 81184


In [56]:
print([col for col in sold.columns if 'invalid' in col or 'missing' in col or 'zero' in col])
print([col for col in listings.columns if 'invalid' in col or 'missing' in col or 'zero' in col])

['invalid_close_price', 'invalid_living_area', 'invalid_days_on_market', 'invalid_bed_or_bath', 'missing_coordinates', 'zero_coordinates', 'invalid_longitude']
['invalid_close_price', 'invalid_living_area', 'invalid_days_on_market', 'invalid_bed_or_bath', 'missing_coordinates', 'zero_coordinates', 'invalid_longitude']


In [57]:
print("=== Complete Flag Summary (Sold) ===")
all_flag_cols = [
    'invalid_close_price', 'invalid_living_area', 'invalid_days_on_market', 'invalid_bed_or_bath',
    'missing_coordinates', 'zero_coordinates', 'invalid_longitude',
    'listing_after_close_flag', 'purchase_after_close_flag', 'negative_timeline_flag',
    'out_of_state_flag', 'out_of_bounds_flag'
]

for col in all_flag_cols:
    if col in sold.columns:
        print(f"{col}: {sold[col].sum()}")

print("\n=== Complete Flag Summary (Listings) ===")
listing_flag_cols = [
    'invalid_living_area', 'invalid_days_on_market', 'invalid_bed_or_bath',
    'missing_coordinates', 'zero_coordinates', 'invalid_longitude',
    'out_of_state_flag', 'out_of_bounds_flag'
]

for col in listing_flag_cols:
    if col in listings.columns:
        print(f"{col}: {listings[col].sum()}")

=== Complete Flag Summary (Sold) ===
invalid_close_price: 1
invalid_living_area: 165
invalid_days_on_market: 51
invalid_bed_or_bath: 0
missing_coordinates: 4388
zero_coordinates: 37
invalid_longitude: 31
listing_after_close_flag: 70
purchase_after_close_flag: 239
negative_timeline_flag: 295
out_of_state_flag: 30
out_of_bounds_flag: 4492

=== Complete Flag Summary (Listings) ===
invalid_living_area: 393
invalid_days_on_market: 32
invalid_bed_or_bath: 0
missing_coordinates: 80824
zero_coordinates: 75
invalid_longitude: 92
out_of_state_flag: 236
out_of_bounds_flag: 81184


In [ ]:
#future - look into flagged columns + investigate if other columns are valuable for our dashboard + look at lat + long columns

#look into listing + closing dates that are in the future

#some huge bathroom + bedroom numbers that need to be looked into 